In [ ]:
import numpy as np
import joblib
import json
import pandas as pd

# Charger modèle et mapping
rf = joblib.load('../models/random_forest_model.pkl')

with open('../data/label_encoding.json') as f:
    mapping = json.load(f)

with open('../data/attck_extended_mapping.json') as f:
    attck_mapping = json.load(f)

# Inverser le mapping id → nom
id_to_label = {v: k for k, v in mapping.items()}
print("Classes :", list(mapping.keys()))

In [ ]:
def score_threat(predicted_label: str, confidence: float,
                 features: dict = None) -> dict:
    """
    Calcule un score de criticité avancé basé sur :
    - Le type d'attaque (score de base ATT&CK)
    - La confiance du modèle
    - Les features de l'événement
    """
    base_info  = attck_mapping.get(predicted_label, {})
    base_score = base_info.get('criticality_score', 0)

    # Modificateurs selon les features
    modifier = 0
    reasons  = []

    if features:
        # Beaucoup de tentatives échouées → plus dangereux
        if features.get('num_failed_logins', 0) > 50:
            modifier += 1
            reasons.append("Nombreuses tentatives échouées (+1)")

        # Gros volume de données → exfiltration possible
        if features.get('src_bytes', 0) > 50000:
            modifier += 1
            reasons.append("Volume de données élevé (+1)")

        # Connexion réussie après attaque → critique
        if features.get('logged_in', 0) == 1 and predicted_label != 'NORMAL':
            modifier += 2
            reasons.append("Connexion réussie après attaque (+2)")

    # Confiance haute = score plus fiable
    confidence_factor = round(confidence * base_score, 2)

    final_score = min(10, base_score + modifier)

    return {
        'base_score':         base_score,
        'modifier':           modifier,
        'final_score':        final_score,
        'confidence_factor':  confidence_factor,
        'alert_level':        _alert_level(final_score),
        'reasons':            reasons
    }

def _alert_level(score):
    if score == 0:  return 'INFO'
    if score <= 4:  return 'LOW'
    if score <= 6:  return 'MEDIUM'
    if score <= 8:  return 'HIGH'
    return 'CRITICAL'

# Test
result = score_threat(
    'BRUTE_FORCE', 0.94,
    features={'num_failed_logins': 75, 'logged_in': 1}
)
print(json.dumps(result, indent=2))

In [ ]:
# Simuler des événements d'un attaquant pour créer son profil
attacker_events = [
    {'source_ip': '192.168.1.45', 'label': 'PORT_SCAN',   'confidence': 0.91, 'timestamp': '2024-05-08T14:00:00'},
    {'source_ip': '192.168.1.45', 'label': 'BRUTE_FORCE', 'confidence': 0.94, 'timestamp': '2024-05-08T14:15:00'},
    {'source_ip': '192.168.1.45', 'label': 'BRUTE_FORCE', 'confidence': 0.88, 'timestamp': '2024-05-08T14:16:00'},
    {'source_ip': '192.168.1.45', 'label': 'WEB_ATTACK',  'confidence': 0.76, 'timestamp': '2024-05-08T14:30:00'},
    {'source_ip': '192.168.1.45', 'label': 'BOT',         'confidence': 0.95, 'timestamp': '2024-05-08T15:00:00'},
]

def build_attacker_profile(events: list) -> dict:
    """Construit le profil complet d'un attaquant depuis ses événements."""
    from collections import Counter

    ip          = events[0]['source_ip']
    labels      = [e['label'] for e in events]
    tactics     = [attck_mapping.get(l, {}).get('att_ck_tactic') for l in labels]
    scores      = [attck_mapping.get(l, {}).get('criticality_score', 0) for l in labels]
    techniques  = [attck_mapping.get(l, {}).get('att_ck_technique_id') for l in labels]

    return {
        'source_ip':           ip,
        'total_events':        len(events),
        'attack_sequence':     labels,
        'tactics_used':        list(set(t for t in tactics if t)),
        'techniques_used':     list(set(t for t in techniques if t)),
        'max_criticality':     max(scores),
        'avg_criticality':     round(sum(scores)/len(scores), 2),
        'most_common_attack':  Counter(labels).most_common(1)[0][0],
        'threat_level':        _alert_level(max(scores)),
        'first_seen':          events[0]['timestamp'],
        'last_seen':           events[-1]['timestamp'],
        'kill_chain_stage':    _identify_kill_chain_stage(labels)
    }

def _identify_kill_chain_stage(labels):
    """Identifie l'étape de la kill chain selon la séquence d'attaques."""
    if 'BOT' in labels:          return 'Stage 7 — Command & Control'
    if 'WEB_ATTACK' in labels:   return 'Stage 3 — Exploitation'
    if 'BRUTE_FORCE' in labels:  return 'Stage 2 — Weaponization'
    if 'PORT_SCAN' in labels:    return 'Stage 1 — Reconnaissance'
    return 'Stage 0 — Unknown'

profile = build_attacker_profile(attacker_events)
print(json.dumps(profile, indent=2))

In [ ]:
import os

scorer_code = '''import json, os

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
with open(os.path.join(BASE_DIR, 'data', 'attck_extended_mapping.json')) as f:
    ATTCK = json.load(f)

def score_threat(label, confidence, features=None):
    info     = ATTCK.get(label, {})
    base     = info.get('criticality_score', 0)
    modifier = 0
    reasons  = []
    if features:
        if features.get('num_failed_logins', 0) > 50:
            modifier += 1; reasons.append("Nombreuses tentatives (+1)")
        if features.get('src_bytes', 0) > 50000:
            modifier += 1; reasons.append("Volume élevé (+1)")
        if features.get('logged_in', 0) == 1 and label != 'NORMAL':
            modifier += 2; reasons.append("Connexion réussie (+2)")
    final = min(10, base + modifier)
    return {
        'base_score': base, 'modifier': modifier,
        'final_score': final, 'reasons': reasons,
        'alert_level': ('INFO' if final==0 else 'LOW' if final<=4
                        else 'MEDIUM' if final<=6 else 'HIGH' if final<=8
                        else 'CRITICAL')
    }
'''

with open('../src/threat_scorer.py', 'w', encoding='utf-8') as f:
    f.write(scorer_code)
print("✅ threat_scorer.py sauvegardé !")